# CDS6334 VIP — Histogram Analysis (Hazy + Low-Light)

This notebook generates grayscale intensity histograms for:
- Raw images
- Student enhanced outputs
- Ground Truth (GT) enhanced images

It saves histogram figures to: `figures/histograms/`

In [21]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [22]:
# -----------------------------
#       Project paths
# -----------------------------
PROJECT_ROOT = Path.cwd().parent

# Find dataset folder (must be = "dataset"")
DATASET_DIR = None
for name in ["dataset"]:
    candidate = PROJECT_ROOT / name
    if candidate.exists():
        DATASET_DIR = candidate
        break

if DATASET_DIR is None:
    raise FileNotFoundError(f"dataset folder not found under: {PROJECT_ROOT}")

OUTPUT_DIR = PROJECT_ROOT / "output"
FIG_DIR = PROJECT_ROOT / "figures" / "histograms"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
#       Dataset folders 
# -----------------------------
HAZY_RAW = DATASET_DIR / "01. Hazy - Raw"
HAZY_GT  = DATASET_DIR / "01. Hazy - Enhanced (GT)"
HAZY_STU = OUTPUT_DIR / "hazy-student-enhanced"

LOW_RAW  = DATASET_DIR / "02. Low Light - Raw"
LOW_GT   = DATASET_DIR / "02. Low Light - Enhanced (GT)"
LOW_STU  = OUTPUT_DIR / "lowlight-student-enhanced"

# Quick checks
for p in [HAZY_RAW, HAZY_GT, HAZY_STU, LOW_RAW, LOW_GT, LOW_STU]:
    if not p.exists():
        print("[MISSING]", p)
    else:
        print("[OK]", p)

[OK] d:\Malaysia\MMU_Studies\Year 3\1st Long Trimester\Visual Information Processing\Assignment\dataset\01. Hazy - Raw
[OK] d:\Malaysia\MMU_Studies\Year 3\1st Long Trimester\Visual Information Processing\Assignment\dataset\01. Hazy - Enhanced (GT)
[OK] d:\Malaysia\MMU_Studies\Year 3\1st Long Trimester\Visual Information Processing\Assignment\output\hazy-student-enhanced
[OK] d:\Malaysia\MMU_Studies\Year 3\1st Long Trimester\Visual Information Processing\Assignment\dataset\02. Low Light - Raw
[OK] d:\Malaysia\MMU_Studies\Year 3\1st Long Trimester\Visual Information Processing\Assignment\dataset\02. Low Light - Enhanced (GT)
[OK] d:\Malaysia\MMU_Studies\Year 3\1st Long Trimester\Visual Information Processing\Assignment\output\lowlight-student-enhanced


## Helper functions

We compute **grayscale histograms** (0–255) because:
- it matches how PSNR/SSIM are commonly evaluated, and
- it is easier to interpret for brightness + contrast changes.

In [23]:
def read_gray(img_path: Path) -> np.ndarray:
    """Read image and return grayscale uint8."""
    img = cv2.imread(str(img_path))
    if img is None:
        raise FileNotFoundError(f"Cannot read image: {img_path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

def plot_gray_hist(gray: np.ndarray, title: str, save_path: Path = None):
    """
    Plot grayscale histogram (256 bins, 0..255).
    If save_path is provided, saves the figure as PNG.
    """
    plt.figure(figsize=(6, 4))
    plt.hist(gray.ravel(), bins=256, range=(0, 255))
    plt.title(title)
    plt.xlabel("Pixel Intensity (0 = black, 255 = white)")
    plt.ylabel("Number of Pixels")
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=200)
    plt.show()
    plt.close()

In [24]:
# -----------------------------
# Create structured output folders
# -----------------------------
H_OUT_RAW     = FIG_DIR / "hazy_raw"
H_OUT_STU     = FIG_DIR / "hazy_student"
H_OUT_GT      = FIG_DIR / "hazy_gt"

L_OUT_RAW     = FIG_DIR / "low_raw"
L_OUT_STU     = FIG_DIR / "low_student"
L_OUT_GT      = FIG_DIR / "low_gt"

for d in [H_OUT_RAW, H_OUT_STU, H_OUT_GT, L_OUT_RAW, L_OUT_STU, L_OUT_GT]:
    d.mkdir(parents=True, exist_ok=True)

print("[OK] Histogram output folders ready.")

[OK] Histogram output folders ready.


## Choose an example image

Pick a filename that exists in **Raw**, **Student output**, and **GT** for both categories.
Example: `9.png`

In [25]:
# -----------------------------
# Selected image names
# -----------------------------

HAZY_IMAGES = [
    "1400.png", "1401.png", "1417.png", "1418.png", "1419.png",
    "1420.png", "1421.png", "1422.png", "1424.png"
]

LOW_IMAGES = [
    "9.png", "73.png", "91.png", "87.png", "731.png",
    "733.png", "99.png", "735.png", "729.png"
]


In [26]:
def save_gray_histogram(img_path: Path, save_path: Path, title: str):
    """
    Generate and save a grayscale histogram for a single image.
    """
    img = cv2.imread(str(img_path))
    if img is None:
        print("[SKIP] Cannot read:", img_path)
        return

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    plt.figure(figsize=(6, 4))
    plt.hist(gray.ravel(), bins=256, range=(0, 255))
    plt.title(title)
    plt.xlabel("Pixel Intensity (0–255)")
    plt.ylabel("Number of Pixels")
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()


In [27]:
def generate_histograms(image_list, raw_dir, stu_dir, gt_dir, label, out_raw, out_stu, out_gt):
    """
    Generate grayscale histograms for raw, student, and GT images,
    saving each group into separate folders.
    Output filename format: <imagename>_histogram.png
    """
    print(f"\nGenerating histograms for {label} images...")

    for name in image_list:
        raw_path = raw_dir / name
        stu_path = stu_dir / name
        gt_path  = gt_dir / name

        if not (raw_path.exists() and stu_path.exists() and gt_path.exists()):
            print("[MISSING]", label, name)
            continue

        save_gray_histogram(
            raw_path,
            out_raw / f"{name}_histogram.png",
            f"{label} RAW Histogram — {name}"
        )

        save_gray_histogram(
            stu_path,
            out_stu / f"{name}_histogram.png",
            f"{label} Student Enhanced Histogram — {name}"
        )

        save_gray_histogram(
            gt_path,
            out_gt / f"{name}_histogram.png",
            f"{label} GT Histogram — {name}"
        )

    print(f"[DONE] {label} histograms generated.")

In [28]:
generate_histograms(
    HAZY_IMAGES,
    HAZY_RAW, HAZY_STU, HAZY_GT,
    label="Hazy",
    out_raw=H_OUT_RAW, out_stu=H_OUT_STU, out_gt=H_OUT_GT
)

generate_histograms(
    LOW_IMAGES,
    LOW_RAW, LOW_STU, LOW_GT,
    label="Low-Light",
    out_raw=L_OUT_RAW, out_stu=L_OUT_STU, out_gt=L_OUT_GT
)

print("\nAll histogram images saved under:")
print(FIG_DIR)


Generating histograms for Hazy images...
[DONE] Hazy histograms generated.

Generating histograms for Low-Light images...
[DONE] Low-Light histograms generated.

All histogram images saved under:
d:\Malaysia\MMU_Studies\Year 3\1st Long Trimester\Visual Information Processing\Assignment\figures\histograms
